# PS2 Image Catalog — Jupyter port

Per-image OCR + summary for each claim folder, plus a whole-claim summary fused from those records. This is the Python/Jupyter port of `ps2_image_catalog.ts` + `ps3_claim_summary.ts`.

**Walks every claim file** (PDF expanded page-by-page, JPGs as-is) and asks a vision-language model to describe each one. For every image / page the model returns:

- `image_type` — X-ray / IVP / KUB / USG / CT / MRI / Coronary Angiogram / Typed report / Stamp / …
- `body_region` — Abdomen / Chest / Coronary tree / etc.
- `modality_view` — PA / Lateral / RAO-Caudal / Transverse / N/A
- `stage_of_care` — pre / intra / post / uncertain
- `ocr` — `{ language, verbatim, english }` (all 12 Indic languages handled)
- `summary`, `key_observations`, `dates_found`, `image_quality`

After the per-image catalog is written, a second text-only call fuses the records into a whole-claim summary (patient, hospital, procedure, key findings, completeness, gaps).

**Two providers, switch via env var `PROVIDER`:**
- `fireworks` (default) — Fireworks AI; needs `FIREWORKS_API_KEY` in `.env`. Defaults to your `qwen3-vl-8b-instruct` deployment.
- `ollama` — opt-in: `PROVIDER=ollama`. Uses local Ollama at `http://localhost:11434/api/chat`, model `qwen3-vl:8b-instruct`.

**Output** — written next to the source PDFs/images inside the claim folder:
```
Claims/<PACKAGE>/<CLAIM>/image_catalog.json
Claims/<PACKAGE>/<CLAIM>/image_catalog.md
Claims/<PACKAGE>/<CLAIM>/claim_summary.md
```

Make sure `pyproject.toml` / `uv` deps are installed: `pymupdf`, `pillow`, `requests`, `json-repair`, `python-dotenv`.

In [1]:
import os, io, json, base64, time, sys, re
from pathlib import Path

import fitz  # PyMuPDF
from PIL import Image, ImageOps
import requests
from json_repair import repair_json
from dotenv import load_dotenv

load_dotenv()  # picks up OLLAMA_API_KEY / FIREWORKS_API_KEY / etc.

True

## Config

Override any of these at the shell or in a cell before running:
```python
os.environ['PROVIDER'] = 'fireworks'
os.environ['MODEL']    = 'accounts/.../deployments/<id>'
```

In [2]:
PROVIDER = os.environ.get('PROVIDER', 'fireworks')  # 'fireworks' (default) or 'ollama'

OLLAMA_URL        = os.environ.get('OLLAMA_URL',    'http://localhost:11434/api/chat')
FIREWORKS_URL     = os.environ.get('FIREWORKS_URL', 'https://api.fireworks.ai/inference/v1/chat/completions')
FIREWORKS_API_KEY = os.environ.get('FIREWORKS_API_KEY')

DEFAULT_MODELS = {
    'ollama':    'qwen3-vl:8b-instruct',
    'fireworks': 'accounts/ajeya-rao-k-eckusf6m/deployments/euufjyfd',  # qwen3-vl-8b-instruct
}
MODEL = os.environ.get('MODEL', DEFAULT_MODELS[PROVIDER])

if PROVIDER == 'fireworks' and not FIREWORKS_API_KEY:
    raise RuntimeError('PROVIDER=fireworks but FIREWORKS_API_KEY is not set in .env')

# Find the project root by looking for the Claims/ sibling folder. This lets
# the notebook run whether the cwd is the project root (Jupyter Lab launched
# from there) OR the code/ folder (VS Code's per-notebook cwd).
def _find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(4):
        if (p / 'Claims').is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = _find_project_root()
CLAIMS_ROOT  = PROJECT_ROOT / 'Claims'

MAX_EDGE    = 1280       # OCR-friendly resolution
JPEG_Q      = 90
PDF_DPI     = 220        # ~220 DPI, readable handwriting / Indic glyphs
NUM_PREDICT = 4096
TEMPERATURE = 0.05
TOP_K       = 30
REPEAT_PEN  = 1.0
SUPPORTED_IMG = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tiff', '.tif'}

print(f'Provider:     {PROVIDER}')
print(f'Endpoint:     {FIREWORKS_URL if PROVIDER == "fireworks" else OLLAMA_URL}')
print(f'Model:        {MODEL}')
print(f'Project root: {PROJECT_ROOT}')

Provider:     fireworks
Endpoint:     https://api.fireworks.ai/inference/v1/chat/completions
Model:        accounts/ajeya-rao-k-eckusf6m/deployments/euufjyfd
Project root: G:\PROJECTS\medical_reports_summary


## System prompt

Verbatim port of the prompt in `ps2_image_catalog.ts`. Keep it identical so both pipelines emit the same JSON schema and reviewer-style language.

In [3]:
SYSTEM = r'''You are a medical document OCR + description assistant for India's
AB PM-JAY claims dataset. You receive ONE image at a time. The image may be:
  - A medical scan (X-ray, IVP, KUB, USG, CT, MRI, coronary angiogram still)
  - A typed clinical / radiology report (English or Indic-language)
  - A handwritten doctor's note or prescription
  - A doctor's stamp, signature block, hospital letterhead
  - A photograph of a cath-lab monitor

ROLE — STRICTLY DESCRIPTIVE
- Do not diagnose. Do not approve / reject. Do not recommend treatment.
- Describe what you SEE. Cite uncertainty when a feature is unclear.

OCR + LANGUAGE
- Transcribe every legible line VERBATIM in its original script for the
  "ocr.verbatim" field.
- Detect the language for "ocr.language": one of English, Hindi, Tamil, Telugu,
  Bengali, Marathi, Gujarati, Kannada, Malayalam, Punjabi, Odia, Urdu,
  Assamese, mixed, or none (if no readable text).
- Translate the verbatim text into English for "ocr.english" (omit fillers,
  preserve clinical terms, drug names, doses, dates).
- When a date is in DD-MM-YYYY / DD.MM.YYYY / YYYY-MM-DD / Indic numerals,
  normalise to DD/MM/YYYY in "dates_found".

IMAGE_TYPE (pick ONE)
  "Chest X-Ray"   "Abdominal X-Ray (KUB)"   "IVP"   "Urethrogram"
  "Ultrasound"    "CT"   "MRI"   "Coronary Angiogram"
  "Typed report"  "Handwritten note"   "Stamp/Signature"
  "Hospital letterhead"   "Patient photo"   "Other"

For "Coronary Angiogram" specifically: cath-lab fluoroscopy frames have a
dark background, grey vessel structures, and on-screen UI overlays such as
"AXIOM-Artis", "RAO/LAO/CRAN/CAUD N°", "EE %", "DDO %", "WC", "WW", "CARD f/s",
or "73/F"-style patient header. ALL such monitor screenshots are
"Coronary Angiogram" — even when guidewires, balloons, or stents are present.

BODY_REGION (free text, anatomically precise)
  e.g. "Chest (PA)", "Abdomen — pelvicalyceal system", "Left kidney + ureter",
       "Coronary tree — left system", "Hepatobiliary system",
       "N/A (text document)"

MODALITY_VIEW (free text)
  e.g. "PA", "Lateral", "Oblique", "RAO 30° / Caudal 20°", "Transverse",
       "Sagittal", "Long-axis", "N/A"

OUTPUT — exactly ONE JSON object, no markdown, no code fence, no prose:

{
  "image_type":     "<from list above>",
  "body_region":    "<string or 'N/A'>",
  "modality_view":  "<string or 'N/A'>",
  "stage_of_care":  "pre-procedure" | "intra-procedure" | "post-procedure" | "uncertain" | "n/a",
  "stage_evidence": "<short clue>",
  "ocr": {
    "language":  "<English | Hindi | Tamil | Telugu | Bengali | Marathi | Gujarati | Kannada | Malayalam | Punjabi | Odia | Urdu | Assamese | mixed | none>",
    "verbatim":  "<every legible line in original script — newline-separated>",
    "english":   "<English translation; same as verbatim if already English>"
  },
  "summary":          "<1-3 sentences: what this image shows>",
  "key_observations": ["<bullet 1>", "<bullet 2>", "<bullet 3>"],
  "dates_found":      ["<DD/MM/YYYY>", "..."],
  "image_quality":    {
    "rating":      "good" | "fair" | "poor",
    "limitations": ["<e.g. partial view, low contrast, photograph of monitor, glare, blur>"]
  }
}

FORBIDDEN
- Diagnosing.
- Inventing text not visible in the image.
- Approving / rejecting claims.
- Markdown, code fences, or prose around the JSON.'''


## Image preprocessing — PDF→JPEG, EXIF rotate, resize, base64

PyMuPDF (`fitz`) replaces `pdf-to-img`; Pillow replaces `sharp`. Same effective behaviour: each page becomes a 220 DPI render, then the longest edge is capped at `MAX_EDGE` and re-encoded as quality-90 JPEG.

In [4]:
def pdf_to_pil_images(pdf_path: Path) -> list[Image.Image]:
    """Render every page of a PDF to a PIL Image at PDF_DPI."""
    doc = fitz.open(pdf_path)
    pages = []
    try:
        for page in doc:
            pix = page.get_pixmap(dpi=PDF_DPI, alpha=False)
            pages.append(Image.frombytes('RGB', (pix.width, pix.height), pix.samples))
    finally:
        doc.close()
    return pages

def optimize_image(img: Image.Image) -> bytes:
    """Honour EXIF rotation, downscale to MAX_EDGE on longest side, JPEG-encode."""
    img = ImageOps.exif_transpose(img)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img.thumbnail((MAX_EDGE, MAX_EDGE), Image.Resampling.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=JPEG_Q, optimize=True)
    return buf.getvalue()

def image_to_b64(img: Image.Image) -> str:
    return base64.b64encode(optimize_image(img)).decode('ascii')

## Model call — provider-agnostic

Both providers share the same prompt and a normalised return shape `{content, prompt_eval_count, eval_count}`. Wire format differs:

- **Ollama** — image goes on the message as `images: [base64]`; raw base64 only, no data URL.
- **Fireworks (OpenAI-compat)** — image goes inside `content[]` as `{type: 'image_url', image_url: {url: 'data:image/jpeg;base64,…'}}`.

In [5]:
def user_instruction(label: str) -> str:
    return (
        f'IMAGE: {label}\n\n'
        'Produce the JSON object described in the system prompt. '
        'OCR every legible line verbatim. Translate non-English text to English. '
        'Describe what is visible — body region, modality, observations. '
        'ONE JSON object. NO PROSE.'
    )

def call_ollama(image_b64: str, label: str) -> dict:
    r = requests.post(OLLAMA_URL, json={
        'model':   MODEL,
        'stream':  False,
        'format':  'json',
        'messages': [
            {'role': 'system', 'content': SYSTEM},
            {'role': 'user',   'content': user_instruction(label), 'images': [image_b64]},
        ],
        'options': {
            'temperature':    TEMPERATURE,
            'top_p':          1,
            'top_k':          TOP_K,
            'repeat_penalty': REPEAT_PEN,
            'num_predict':    NUM_PREDICT,
        },
    }, timeout=300)
    r.raise_for_status()
    j = r.json()
    return {
        'content':           j.get('message', {}).get('content', ''),
        'prompt_eval_count': j.get('prompt_eval_count'),
        'eval_count':        j.get('eval_count'),
    }

def call_fireworks(image_b64: str, label: str) -> dict:
    r = requests.post(FIREWORKS_URL,
        headers={
            'Authorization': f'Bearer {FIREWORKS_API_KEY}',
            'Content-Type':  'application/json',
        },
        json={
            'model': MODEL,
            'messages': [
                {'role': 'system', 'content': SYSTEM},
                {'role': 'user',   'content': [
                    {'type': 'text',      'text': user_instruction(label)},
                    {'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{image_b64}'}},
                ]},
            ],
            'response_format': {'type': 'json_object'},
            'temperature':     TEMPERATURE,
            'top_p':           1,
            'top_k':           TOP_K,
            'max_tokens':      NUM_PREDICT,
        },
        timeout=300,
    )
    r.raise_for_status()
    j = r.json()
    usage = j.get('usage', {}) or {}
    return {
        'content':           j['choices'][0]['message']['content'],
        'prompt_eval_count': usage.get('prompt_tokens'),
        'eval_count':        usage.get('completion_tokens'),
    }

def call_model(image_b64: str, label: str, retries: int = 3) -> dict:
    last = None
    for attempt in range(retries):
        try:
            return (call_fireworks if PROVIDER == 'fireworks' else call_ollama)(image_b64, label)
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else None
            if code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                wait = 2 ** attempt + 1
                print(f'    ⚠  HTTP {code}; retry in {wait}s')
                time.sleep(wait); last = e; continue
            raise
    raise last  # type: ignore[misc]

## Robust JSON parse

Strips code fences, slices to the outermost `{...}`, falls back to `json_repair` if the model emits anything malformed.

In [6]:
def parse_json_obj(text: str) -> dict:
    s = text.strip()
    m = re.match(r'```(?:json)?\s*([\s\S]*?)\s*```', s, re.IGNORECASE)
    if m:
        s = m.group(1).strip()
    start, end = s.find('{'), s.rfind('}')
    if start != -1 and end > start:
        s = s[start:end+1]
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return json.loads(repair_json(s))

## Walk a claim folder → list of (source, page, total, PIL image)

In [7]:
def list_images_in_claim(claim_dir: Path) -> list[tuple[str, int | None, int | None, Image.Image]]:
    files = sorted([
        f for f in claim_dir.iterdir()
        if f.is_file() and (f.suffix.lower() in SUPPORTED_IMG or f.suffix.lower() == '.pdf')
    ])
    out: list[tuple[str, int | None, int | None, Image.Image]] = []
    for f in files:
        if f.suffix.lower() == '.pdf':
            pages = pdf_to_pil_images(f)
            for i, img in enumerate(pages):
                out.append((f.name, i + 1, len(pages), img))
        else:
            img = Image.open(f); img.load()
            out.append((f.name, None, None, img))
    return out

## Markdown summary renderer

Identical layout to the Bun version: an overview table at the top, then a per-image deep section with summary, key observations, dates, and a collapsed verbatim OCR block.

In [8]:
def render_markdown(claim_id: str, package_folder: str, records: list[dict]) -> str:
    L: list[str] = []
    L.append(f'# Image Catalog — {claim_id}'); L.append('')
    L.append(f'**Package:** {package_folder}  ·  **Model:** {MODEL}  ·  **Images:** {len(records)}'); L.append('')

    L.append('## Overview'); L.append('')
    L.append('| # | Source | Page | Type | Body region | Stage | Quality |')
    L.append('|---|--------|------|------|-------------|-------|---------|')
    for r in records:
        if 'error' in r:
            L.append(f"| {r['image_index']} | {r['source']} | {r.get('page') or '—'} | _error_ | — | — | — |")
            continue
        pg = f"{r['page']}/{r['pages_total']}" if r.get('page') else '—'
        rating = (r.get('image_quality') or {}).get('rating', '?')
        L.append(
            f"| {r['image_index']} | {r['source']} | {pg} | {r.get('image_type','?')} | "
            f"{r.get('body_region','?')} | {r.get('stage_of_care','?')} | {rating} |"
        )
    L.append('')

    for r in records:
        pg = f" — page {r['page']}/{r['pages_total']}" if r.get('page') else ''
        L.append(f"## [{r['image_index']}] {r['source']}{pg}"); L.append('')
        if 'error' in r:
            L.append(f"> **Error:** {r['error']}"); L.append(''); continue

        meta: list[str] = []
        if r.get('image_type'):  meta.append(f"**Type:** {r['image_type']}")
        if r.get('body_region'): meta.append(f"**Body region:** {r['body_region']}")
        if r.get('modality_view') and r['modality_view'] != 'N/A':
            meta.append(f"**View:** {r['modality_view']}")
        if r.get('stage_of_care'):
            ev = f" ({r['stage_evidence']})" if r.get('stage_evidence') else ''
            meta.append(f"**Stage:** {r['stage_of_care']}{ev}")
        q = r.get('image_quality') or {}
        if q.get('rating'):
            lim = [x for x in (q.get('limitations') or []) if x]
            extra = f" — {', '.join(lim)}" if lim else ''
            meta.append(f"**Quality:** {q['rating']}{extra}")
        if meta: L.append('  ·  '.join(meta)); L.append('')

        if r.get('summary'): L.append(f"**Summary.** {r['summary']}"); L.append('')

        kobs = r.get('key_observations') or []
        if kobs:
            L.append('**Key observations:**')
            for o in kobs: L.append(f'- {o}')
            L.append('')

        dates = r.get('dates_found') or []
        if dates: L.append(f"**Dates found:** {', '.join(dates)}"); L.append('')

        ocr = r.get('ocr') or {}
        verb = (ocr.get('verbatim') or '').strip()
        eng  = (ocr.get('english') or '').strip()
        if verb or eng:
            L.append(f"**OCR** _(language: {ocr.get('language','?')})_")
            if verb:
                L.append(''); L.append('<details><summary>Verbatim (original script)</summary>'); L.append('')
                L.append('```'); L.append(verb); L.append('```'); L.append('')
                L.append('</details>')
            if eng and eng != verb:
                L.append(''); L.append('**English:**'); L.append('')
                L.append('```'); L.append(eng); L.append('```')
            L.append('')
        L.append('---'); L.append('')
    return '\n'.join(L)

## Claim summary — fuse per-image records into one whole-claim narrative

After the per-image catalog is written, a second text-only call to the same model rolls every record up into a single `claim_summary.md`: patient / hospital / encounter, image inventory by type, a short clinical narrative, key findings, completeness flags, and any gaps.

The summariser only needs `image_catalog.json` — the markdown is a redundant view of the same data. It applies a few defensive tricks:

- **Compact the catalog** before sending: dedupe `key_observations`/`dates_found`, trim long OCR blocks. Repeated OCR junk otherwise tempts the model into n-gram loops.
- **`frequency_penalty: 0.4`** on the Fireworks call as a second guard against loops.
- **Key normalisation** after parsing: the model occasionally emits `Clinical_narrative` / `kay_findings` / `is_*_present` instead of the canonical names, so we map common variants back.

Output: `Claims/<PACKAGE>/<CLAIM>/claim_summary.md` only — no JSON.

In [9]:
SUMMARY_SYSTEM = r"""You are a medical-claim summarisation assistant for India's
AB PM-JAY claims dataset. You receive a JSON list of per-image OCR + description
records produced by a vision model for ONE claim folder. Your job is to fuse
those records into a single whole-claim summary.

ROLE — STRICTLY DESCRIPTIVE
- Do not diagnose. Do not approve / reject. Do not recommend treatment.
- Describe what the dossier shows, gaps included.
- When uncertain, say so. Never invent identifiers, dates, or findings.

INPUTS YOU CAN USE
- Each record has: source filename, page, image_type, body_region, modality_view,
  stage_of_care, summary, key_observations, dates_found, image_quality,
  and an OCR block (verbatim + english).
- Combine OCR text + image observations to identify patient name, age, sex,
  hospital, doctor, dates, procedure performed, and overall narrative.
- Treat OCR field 'english' as the primary text source. Use 'verbatim' to
  resolve disagreements or to capture proper nouns / IDs.

OUTPUT — exactly ONE JSON object, no markdown, no code fence, no prose:

{
  "patient": {
    "name":      "<string or 'unknown'>",
    "age":       "<e.g. '54 years' or 'unknown'>",
    "sex":       "<Male | Female | unknown>",
    "id_numbers":["<any patient/MRN/UHID/Aadhaar-style numbers found>"]
  },
  "hospital": {
    "name":     "<string or 'unknown'>",
    "location": "<city / state if visible, else 'unknown'>",
    "doctors":  ["<doctor names with degrees if visible>"]
  },
  "encounter": {
    "date_range":          "<earliest to latest DD/MM/YYYY found, or 'unknown'>",
    "all_dates":           ["<every distinct DD/MM/YYYY>"],
    "primary_procedure":   "<short phrase, e.g. 'Coronary angiography + PCI to LAD' or 'unknown'>",
    "package_code":        "<PMJAY / scheme code if visible, else 'unknown'>"
  },
  "image_inventory": {
    "total_images":        <int>,
    "by_type":             { "<image_type>": <count>, "...": <count> },
    "stages_present":      ["<pre-procedure | intra-procedure | post-procedure | uncertain | n/a>"],
    "languages_seen":      ["<English | Hindi | ...>"]
  },
  "clinical_narrative": "<2-4 sentence plain-English description of what the dossier portrays — patient context, what was done, what the imaging shows>",
  "key_findings":       ["<bullet — synthesised across images>", "..."],
  "completeness": {
    "has_pre_procedure_imaging":   <true|false>,
    "has_intra_procedure_imaging": <true|false>,
    "has_post_procedure_imaging":  <true|false>,
    "has_typed_report":            <true|false>,
    "has_handwritten_notes":       <true|false>,
    "has_signed_stamp":            <true|false>,
    "notes":                       "<any caveats, e.g. 'no post-procedure imaging present'>"
  },
  "concerns_or_gaps": ["<missing piece, illegible doc, contradiction between records, low-quality scan, etc.>"]
}

FORBIDDEN
- Diagnosing.
- Approving / rejecting the claim.
- Inventing names, IDs, or dates that do not appear in the records.
- Markdown, code fences, or prose around the JSON."""

SUMMARY_TEMPERATURE = 0.1
SUMMARY_NUM_PREDICT = 4096
MAX_LIST_PER_RECORD = 12
MAX_OCR_CHARS       = 2000

def _dedup_cap(arr, max_n=MAX_LIST_PER_RECORD):
    if not isinstance(arr, list): return []
    seen, out = set(), []
    for v in arr:
        s = str(v if v is not None else '').strip()
        if not s or s in seen: continue
        seen.add(s); out.append(s)
        if len(out) >= max_n: break
    return out

def _trim_text(s, max_n=MAX_OCR_CHARS):
    if not isinstance(s, str): return ''
    collapsed = re.sub(r'[ \t]+', ' ', s)
    collapsed = re.sub(r'\n{3,}', '\n\n', collapsed).strip()
    return collapsed if len(collapsed) <= max_n else collapsed[:max_n] + '…[truncated]'

def compact_catalog(catalog: dict) -> dict:
    """Strip _usage / stage_evidence and dedupe noisy fields so the summariser
    isn't primed to loop on repeated OCR junk."""
    records = []
    for r in (catalog.get('records') or []):
        if 'error' in r:
            records.append({'image_index': r.get('image_index'), 'source': r.get('source'),
                            'page': r.get('page'), 'error': r.get('error')})
            continue
        ocr = r.get('ocr') or {}
        records.append({
            'image_index':      r.get('image_index'),
            'source':           r.get('source'),
            'page':             r.get('page'),
            'pages_total':      r.get('pages_total'),
            'image_type':       r.get('image_type'),
            'body_region':      r.get('body_region'),
            'modality_view':    r.get('modality_view'),
            'stage_of_care':    r.get('stage_of_care'),
            'summary':          _trim_text(r.get('summary'), 600),
            'key_observations': _dedup_cap(r.get('key_observations')),
            'dates_found':      _dedup_cap(r.get('dates_found')),
            'ocr': {
                'language': ocr.get('language'),
                'verbatim': _trim_text(ocr.get('verbatim')),
                'english':  _trim_text(ocr.get('english')),
            },
            'image_quality':    r.get('image_quality'),
        })
    return {
        'claim_id': catalog.get('claim_id'),
        'package':  catalog.get('package'),
        'n_images': catalog.get('n_images') or len(records),
        'records':  records,
    }

def _call_text_fireworks(user_text: str) -> str:
    r = requests.post(FIREWORKS_URL,
        headers={'Authorization': f'Bearer {FIREWORKS_API_KEY}', 'Content-Type': 'application/json'},
        json={
            'model': MODEL,
            'messages': [
                {'role': 'system', 'content': SUMMARY_SYSTEM},
                {'role': 'user',   'content': user_text},
            ],
            'response_format':   {'type': 'json_object'},
            'temperature':       SUMMARY_TEMPERATURE,
            'max_tokens':        SUMMARY_NUM_PREDICT,
            'frequency_penalty': 0.4,   # suppress n-gram loops on repetitive OCR noise
            'presence_penalty':  0.1,
        },
        timeout=300,
    )
    r.raise_for_status()
    return r.json()['choices'][0]['message']['content']

def _call_text_ollama(user_text: str) -> str:
    r = requests.post(OLLAMA_URL, json={
        'model':  MODEL,
        'stream': False,
        'format': 'json',
        'messages': [
            {'role': 'system', 'content': SUMMARY_SYSTEM},
            {'role': 'user',   'content': user_text},
        ],
        'options': {'temperature': SUMMARY_TEMPERATURE, 'num_predict': SUMMARY_NUM_PREDICT},
    }, timeout=300)
    r.raise_for_status()
    return r.json().get('message', {}).get('content', '')

def call_text_model(user_text: str, retries: int = 3) -> str:
    last = None
    for attempt in range(retries):
        try:
            return (_call_text_fireworks if PROVIDER == 'fireworks' else _call_text_ollama)(user_text)
        except requests.HTTPError as e:
            code = e.response.status_code if e.response is not None else None
            if code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                wait = 2 ** attempt + 1
                print(f'    ⚠  HTTP {code}; retry in {wait}s')
                time.sleep(wait); last = e; continue
            raise
    raise last  # type: ignore[misc]

def _pick_key(obj, *candidates):
    if not isinstance(obj, dict): return None
    for k in candidates:
        if k in obj: return obj[k]
    lower = {k.lower(): k for k in obj.keys()}
    for k in candidates:
        hit = lower.get(k.lower())
        if hit is not None: return obj[hit]
    return None

def _to_bool(v):
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return v != 0
    if isinstance(v, str):
        s = v.strip().lower()
        if s in ('true', 'yes', 'y', '1', 'present'):       return True
        if s in ('false', 'no', 'n', '0', 'absent', 'none'): return False
    return None

def normalize_summary(raw: dict) -> dict:
    """Map common drift in model-generated keys back to our canonical schema."""
    c = _pick_key(raw, 'completeness') or {}
    return {
        'patient':            _pick_key(raw, 'patient') or {},
        'hospital':           _pick_key(raw, 'hospital') or {},
        'encounter':          _pick_key(raw, 'encounter') or {},
        'image_inventory':    _pick_key(raw, 'image_inventory', 'imageInventory') or {},
        'clinical_narrative': _pick_key(raw, 'clinical_narrative', 'Clinical_narrative', 'clinicalNarrative', 'narrative') or '',
        'key_findings':       _pick_key(raw, 'key_findings', 'kay_findings', 'keyFindings', 'findings') or [],
        'completeness': {
            'has_pre_procedure_imaging':   _to_bool(_pick_key(c, 'has_pre_procedure_imaging',   'is_pre_procedure_imaging_present',   'pre_procedure_imaging')),
            'has_intra_procedure_imaging': _to_bool(_pick_key(c, 'has_intra_procedure_imaging', 'is_intra_procedure_imaging_present', 'intra_procedure_imaging')),
            'has_post_procedure_imaging':  _to_bool(_pick_key(c, 'has_post_procedure_imaging',  'is_post_procedure_imaging_present',  'post_procedure_imaging')),
            'has_typed_report':            _to_bool(_pick_key(c, 'has_typed_report',            'is_typed_report_present',            'typed_report')),
            'has_handwritten_notes':       _to_bool(_pick_key(c, 'has_handwritten_notes',       'is_handwritten_notes_present',       'handwritten_notes')),
            'has_signed_stamp':            _to_bool(_pick_key(c, 'has_signed_stamp',            'is_signed_stamp_present',            'signed_stamp', 'has_stamp')),
            'notes':                       _pick_key(c, 'notes', 'note') or '',
        },
        'concerns_or_gaps':   _pick_key(raw, 'concerns_or_gaps', 'gaps_or_concerns', 'concerns', 'gaps') or [],
    }

def render_summary_markdown(s: dict) -> str:
    L: list[str] = []
    L.append(f"# Claim Summary — {s['claim_id']}"); L.append('')
    L.append(f"**Package:** {s['package']}  ·  **Model:** {s['model']}  ·  **Images:** {s['n_images']}"); L.append('')

    p = s.get('patient') or {}
    L.append('## Patient')
    L.append(f"- **Name:** {p.get('name', 'unknown')}")
    L.append(f"- **Age:** {p.get('age', 'unknown')}")
    L.append(f"- **Sex:** {p.get('sex', 'unknown')}")
    if isinstance(p.get('id_numbers'), list) and p['id_numbers']:
        L.append(f"- **IDs:** {', '.join(p['id_numbers'])}")
    L.append('')

    h = s.get('hospital') or {}
    L.append('## Hospital')
    L.append(f"- **Name:** {h.get('name', 'unknown')}")
    L.append(f"- **Location:** {h.get('location', 'unknown')}")
    if isinstance(h.get('doctors'), list) and h['doctors']:
        L.append(f"- **Doctors:** {'; '.join(h['doctors'])}")
    L.append('')

    e = s.get('encounter') or {}
    L.append('## Encounter')
    L.append(f"- **Date range:** {e.get('date_range', 'unknown')}")
    L.append(f"- **Primary procedure:** {e.get('primary_procedure', 'unknown')}")
    L.append(f"- **Package code:** {e.get('package_code', 'unknown')}")
    if isinstance(e.get('all_dates'), list) and e['all_dates']:
        L.append(f"- **All dates seen:** {', '.join(e['all_dates'])}")
    L.append('')

    if s.get('clinical_narrative'):
        L.append('## Clinical narrative'); L.append('')
        L.append(s['clinical_narrative']); L.append('')

    if isinstance(s.get('key_findings'), list) and s['key_findings']:
        L.append('## Key findings')
        for f in s['key_findings']: L.append(f'- {f}')
        L.append('')

    inv = s.get('image_inventory') or {}
    L.append('## Image inventory')
    L.append(f"- **Total images:** {inv.get('total_images', s['n_images'])}")
    if isinstance(inv.get('by_type'), dict) and inv['by_type']:
        L.append('- **By type:**')
        for k, v in inv['by_type'].items(): L.append(f'  - {k}: {v}')
    if isinstance(inv.get('stages_present'), list) and inv['stages_present']:
        L.append(f"- **Stages present:** {', '.join(inv['stages_present'])}")
    if isinstance(inv.get('languages_seen'), list) and inv['languages_seen']:
        L.append(f"- **Languages:** {', '.join(inv['languages_seen'])}")
    L.append('')

    c = s.get('completeness') or {}
    L.append('## Completeness')
    L.append(f"- Pre-procedure imaging: {c.get('has_pre_procedure_imaging', '?')}")
    L.append(f"- Intra-procedure imaging: {c.get('has_intra_procedure_imaging', '?')}")
    L.append(f"- Post-procedure imaging: {c.get('has_post_procedure_imaging', '?')}")
    L.append(f"- Typed report: {c.get('has_typed_report', '?')}")
    L.append(f"- Handwritten notes: {c.get('has_handwritten_notes', '?')}")
    L.append(f"- Signed stamp: {c.get('has_signed_stamp', '?')}")
    if c.get('notes'): L.append(f"- **Notes:** {c['notes']}")
    L.append('')

    if isinstance(s.get('concerns_or_gaps'), list) and s['concerns_or_gaps']:
        L.append('## Concerns / gaps')
        for g in s['concerns_or_gaps']: L.append(f'- {g}')
        L.append('')

    return '\n'.join(L)

def summarize_claim(package_folder: str, claim_id: str) -> dict | None:
    """Read image_catalog.json from the claim folder, ask the model for a
    whole-claim summary, write claim_summary.md alongside."""
    claim_dir   = CLAIMS_ROOT / package_folder / claim_id
    catalog_path = claim_dir / 'image_catalog.json'
    if not catalog_path.exists():
        print(f'  ⚠  no image_catalog.json — skip summary ({catalog_path})')
        return None

    print(f'\n→ summarise {package_folder}/{claim_id}')
    catalog = json.loads(catalog_path.read_text(encoding='utf-8'))
    compact = compact_catalog(catalog)

    user_text = (
        f'Claim: {package_folder}/{claim_id}\n'
        'Per-image catalog (JSON below). Produce the whole-claim summary JSON object '
        'described in the system prompt. ONE JSON object. NO PROSE.\n\n'
        + json.dumps(compact, indent=2, ensure_ascii=False)
    )

    t0  = time.time()
    raw = call_text_model(user_text)
    dt  = int((time.time() - t0) * 1000)
    obj = normalize_summary(parse_json_obj(raw or '{}'))

    out = {
        'claim_id':     catalog.get('claim_id') or claim_id,
        'package':      catalog.get('package')  or package_folder,
        'model':        MODEL,
        'n_images':     catalog.get('n_images') or compact['n_images'],
        'generated_ms': dt,
        **obj,
    }

    md_path = claim_dir / 'claim_summary.md'
    md_path.write_text(render_summary_markdown(out), encoding='utf-8')
    print(f'  ✓ {md_path.resolve()}  ({dt} ms)')
    return out

## Main pipeline — `catalog_claim` and `catalog_package`

In [10]:
def catalog_claim(package_folder: str, claim_id: str, limit: int | None = None) -> dict | None:
    claim_dir = CLAIMS_ROOT / package_folder / claim_id
    if not claim_dir.is_dir():
        raise RuntimeError(f'Not a folder: {claim_dir}')

    print(f'\n→ {package_folder}/{claim_id}')
    images = list_images_in_claim(claim_dir)
    if limit: images = images[:limit]
    if not images: print('  no images'); return None
    print(f'  {len(images)} image(s) total')

    records: list[dict] = []
    for idx, (source, page, total, img) in enumerate(images):
        label = f'{source} page {page}/{total}' if page else source
        sys.stdout.write(f'  [{idx+1}/{len(images)}] {label} … '); sys.stdout.flush()
        try:
            b64  = image_to_b64(img)
            t0   = time.time()
            resp = call_model(b64, label)
            dt   = int((time.time() - t0) * 1000)
            obj  = parse_json_obj(resp['content'] or '{}')
            rec = {
                'image_index': idx,
                'source':      source,
                'page':        page,
                'pages_total': total,
                **obj,
                '_usage': {
                    'prompt_tokens':     resp['prompt_eval_count'],
                    'completion_tokens': resp['eval_count'],
                    'total_duration_ms': dt,
                },
            }
            records.append(rec)
            print(f"{obj.get('image_type', '?')}  ({dt} ms)")
        except Exception as e:
            print(f'ERROR — {e}')
            records.append({'image_index': idx, 'source': source, 'page': page, 'error': str(e)})

    out = {
        'claim_id': claim_id,
        'package':  package_folder,
        'model':    MODEL,
        'n_images': len(records),
        'records':  records,
    }

    # Write the catalog next to the source files inside the claim folder.
    json_path = claim_dir / 'image_catalog.json'
    json_path.write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'  ✓ {json_path.resolve()}')

    md_path = claim_dir / 'image_catalog.md'
    md_path.write_text(render_markdown(claim_id, package_folder, records), encoding='utf-8')
    print(f'  ✓ {md_path.resolve()}')

    # Roll the per-image records up into a whole-claim summary.
    try:
        summarize_claim(package_folder, claim_id)
    except Exception as e:
        print(f'  ⚠  claim summary failed: {e}')

    return out

def catalog_package(package_folder: str, limit: int | None = None) -> None:
    pkg_dir = CLAIMS_ROOT / package_folder
    if not pkg_dir.is_dir(): raise RuntimeError(f'No folder: {pkg_dir}')
    claims = sorted([d.name for d in pkg_dir.iterdir() if d.is_dir()])
    for c in claims:
        try: catalog_claim(package_folder, c, limit=limit)
        except Exception as e: print(f'  ✗ {c}: {e}')

## Run — single claim

Edit `PACKAGE` / `CLAIM` and run the cell. Set `limit=3` while iterating to keep cost low.

In [13]:
PACKAGE = 'MG029A'
CLAIM   = 'MAV_GJ_R3_2026033110059264'

result = catalog_claim(PACKAGE, CLAIM)  # add limit=3 to test cheap


→ MG029A/MAV_GJ_R3_2026033110059264
  2 image(s) total
  [1/2] 000021__MAV_GJ_R3_2026033110059264__X_RAY_REPORT.pdf page 1/1 … Typed report  (4320 ms)
  [2/2] 000063__MAV_GJ_R3_2026033110059264__X_RAY_PLATE.jpeg … Chest X-Ray  (2945 ms)
  ✓ G:\PROJECTS\medical_reports_summary\Claims\MG029A\MAV_GJ_R3_2026033110059264\image_catalog.json
  ✓ G:\PROJECTS\medical_reports_summary\Claims\MG029A\MAV_GJ_R3_2026033110059264\image_catalog.md

→ summarise MG029A/MAV_GJ_R3_2026033110059264
  ✓ G:\PROJECTS\medical_reports_summary\Claims\MG029A\MAV_GJ_R3_2026033110059264\claim_summary.md  (3571 ms)


### Inspect a single record without scrolling the JSON

In [12]:
if result and result['records']:
    r = result['records'][0]
    print(f"Image #{r['image_index']} — {r['source']}" + (f" page {r['page']}/{r['pages_total']}" if r.get('page') else ''))
    print(f"Type: {r.get('image_type')} · Body: {r.get('body_region')} · Stage: {r.get('stage_of_care')}")
    print('Summary:', r.get('summary'))
    print('Dates:',   r.get('dates_found'))

Image #0 — 000019__MMJAA_UP_2026_R3_2026033010053802__MALATIKUV.pdf page 1/2
Type: Typed report · Body: Abdomen — pelvicalyceal system · Stage: n/a
Summary: This is a typed radiology report for an X-ray KUB (Kidney, Ureter, Bladder) performed on a 36-year-old female. The report notes the absence of significant radioopacity in the renal fossa, normal appearance of surrounding soft tissues and fat shadows, and confirms the presence of a DJ stent in the left side with its ends positioned in the renal region and pelvis, respectively. The impression is 'Normal Study' with advice for clinical correlation.
Dates: ['02/04/2026']


## Run — every claim in a package

Each iteration writes its own `image_catalog.json` and `.md` inside that claim's folder.

In [13]:
# catalog_package('SU007A')
# catalog_package('MC011A')
# catalog_package('MG029A')
# catalog_package('SG039')   # alias for SG039C — folder is named SG039 in this repo

## Switch to local Ollama mid-session

Re-run the **Config** cell after changing the env var.

In [14]:
# os.environ['PROVIDER'] = 'ollama'
# os.environ.pop('MODEL', None)   # use the Ollama default (qwen3-vl:8b-instruct)
# # then re-run the config cell + run cell